### Diffusion of a gaussian packegue

Number density of each flavor component along $x$. Each curve is one snapshot; color is time.

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import h5py
import numpy as np

try:
    here = Path(__file__).resolve().parent
except NameError:
    here = Path.cwd()

sys.path.append(str((here / '..' / 'plot_functions').resolve()))
sys.path.append(str((here / '..' / 'phys_const').resolve()))

import plot_functions as pf
import phys_const as pc

Gaussian diffusion test simulation paramters

In [ ]:
# Directory path
directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test'

directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test_ncell_1024_1_1_1_dts_1.0e-12'
ncell = (1024, 1, 1)

directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test_ncell_768_1_1_1_dts_1.0e-12'
ncell = (768, 1, 1)

directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test_ncell_512_1_1_1_dts_1.0e-12'
ncell = (512, 1, 1)

directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test_ncell_256_1_1_1_dts_1.0e-12'
ncell = (256, 1, 1)

directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test_ncell_128_1_1_1_dts_1.0e-12'
ncell = (128, 1, 1)

number_of_flavors = 3

# Simulation domain
Lx = 200.0 # cm
Ly = 0.1953125 # cm
Lz = 0.1953125  # cm
dx = Lx / ncell[0]
dy = Ly / ncell[1]
dz = Lz / ncell[2]
domain = (Lx, Ly, Lz)  # cm
x_cell_centers = np.arange(domain[0] / ncell[0] / 2, domain[0], domain[0] / ncell[0])

# Scattering opacities
kappar_cm = 5.91e-01
kapparbar_cm = 5.91e-01

# Initial conditions of Gaussian packet
# n = n_bg + A0 * exp(-(x - x0)^2 / (2 s0^2))
n_bg_ccm = 1.0e28
A0_ccm = 1.0e29
x0_cm = 100.0
s0_cm = 5.0

# Snapshots drawn on each spatial plot
lowlimfit = None
uplimfit = None

Analitic solution

In [ ]:
att_num_diff = 0.0

D_cm2_s = pc.PhysConst.c / (3.0 * kappar_cm)
D_num_cm2_s = pc.PhysConst.c * kappar_cm * dx**2 / 4.0
D_num_cm2_s *= att_num_diff

print(f'D_cm2_s (physical diffusion) = {D_cm2_s:.2e}')
print(f'D_num_cm2_s (numerical diffusion) = {D_num_cm2_s:.2e}')

Dnum_bigger_D = D_num_cm2_s > D_cm2_s

if Dnum_bigger_D:
    print('numerical diffusion is bigger than physical diffusion!')
    minval = 1.0e-2
    k_ideal  = np.sqrt( 4*minval / (dx     **2 * 3))
    dx_ideal = np.sqrt( 4*minval / (k_ideal**2 * 3))
    print(f"dx (previous): {dx:.2e}, dx_ideal (ideal): {dx_ideal:.2e}")
    print(f"kappar_cm (previous): {kappar_cm:.2e}, k_ideal_cm (ideal): {k_ideal:.2e}")    
else:
    print('physical diffusion is bigger than numerical diffusion!')

D_cm2_s += D_num_cm2_s

def analytic_solution(t_s, x_cm=None):
    """Cartesian reference density n(x, t) in cm^-3.

    n(x, t) = n_bg + A0 * s0 / sqrt(s0^2 + 2 D t)
              * exp(-(x - x0)^2 / (2 (s0^2 + 2 D t)))

    t_s is a time in seconds, or an array of times. x_cm defaults to the
    cell centers of the run. A scalar time returns shape (nx,); an array
    of times returns shape (nt, nx).
    """
    if x_cm is None:
        dx = domain[0] / ncell[0]
        x_cm = np.arange(dx / 2.0, domain[0], dx)
    x_cm = np.asarray(x_cm, dtype=float)
    t_in = np.asarray(t_s, dtype=float)
    variance = s0_cm**2 + 2.0 * D_cm2_s * np.atleast_1d(t_in)
    n = (
        n_bg_ccm
        + A0_ccm * s0_cm / np.sqrt(variance)[:, None]
        * np.exp(-((x_cm - x0_cm) ** 2) / (2.0 * variance[:, None]))
    )
    if t_in.ndim == 0:
        return n[0]
    return n

Reading simulation data

In [ ]:
all_data_path = os.path.join(directory_path, 'allData.h5')
with h5py.File(all_data_path, 'r') as all_data_file:
    nee = np.array(all_data_file['/N00_Re(1|ccm)'][:])
    nuu = np.array(all_data_file['/N11_Re(1|ccm)'][:])
    neu = np.sqrt(
        np.array(all_data_file['/N01_Re(1|ccm)'][:])**2
        + np.array(all_data_file['/N01_Im(1|ccm)'][:])**2
    )
    if number_of_flavors == 3:
        ntt = np.array(all_data_file['/N22_Re(1|ccm)'][:])
        net = np.sqrt(
            np.array(all_data_file['/N02_Re(1|ccm)'][:])**2
            + np.array(all_data_file['/N02_Im(1|ccm)'][:])**2
        )
        nut = np.sqrt(
            np.array(all_data_file['/N12_Re(1|ccm)'][:])**2
            + np.array(all_data_file['/N12_Im(1|ccm)'][:])**2
        )
    time_s = np.array(all_data_file['/t(s)'][:])

print(f"nee.shape: {nee.shape}")
print(f"time_s.shape: {time_s.shape}")


Plotting data

In [ ]:
pf.plot_colored_lines(
    x_cell_centers,                         # x-axis: space/grid centers, shape (1024,)
    nee[:, :, 0, 0],                      # y values: need shape (space, time)
    time_s,                                 # t-axis: list of times
    r'$z \, [\mathrm{cm}]$',                # x label
    r'$n_{ee} \, [\mathrm{cm}^{-3}$]',      # y label
    r'$t \, [s]$',                          # t label
    f"plots/{directory_path.split('/')[-1]}_n_ee_space.pdf",
    xlog= False, 
    ylog=False, 
    ylowerlimit=None, 
    yupperlimit=None,
    xlowerlimit=100.0,
    xupperlimit=140.0
)


Error stimation

In [ ]:
error_estimate_at_time = np.zeros_like(nee[:, :, 0, 0])
analytic_solution_all_times = np.zeros_like(nee[:, :, 0, 0])
for i in range(len(time_s)):
    analytic_solution_at_time = analytic_solution(time_s[i], x_cell_centers)
    error_estimate_at_time[i, :] = np.abs(nee[i, :, 0, 0] - analytic_solution_at_time)/analytic_solution_at_time
    analytic_solution_all_times[i, :] = analytic_solution_at_time

pf.plot_colored_lines(
    x_cell_centers,                         # x-axis: space/grid centers, shape (1024,)
    np.log10(error_estimate_at_time),                      # y values: need shape (space, time)
    time_s,                                 # t-axis: list of times
    r'$z \, [\mathrm{cm}]$',                # x label
    r'$\log_{10}\,(\,\mathrm{relative\,error}\,)$',      # y label
    r'$t \, [s]$',                          # t label
    f"plots/{directory_path.split('/')[-1]}_n_ee_error_space.pdf",
    xlog= False, 
    ylog= False, 
    ylowerlimit=None, 
    yupperlimit=None,
    xlowerlimit=100.0,
    xupperlimit=140.0
)


Conservation of number of neutrinos

In [ ]:
number_of_neutrinos = np.zeros_like(time_s)
for i in range(len(time_s)):
    number_of_neutrinos[i] = np.sum(nee[i, :, 0, 0]*dx*dy*dz)

plt.figure(figsize=(8,6))
plt.plot(time_s, number_of_neutrinos/number_of_neutrinos[0] - 1, linewidth=2)
plt.xlabel(r'$t \, [s]$')
plt.ylabel(r'$N_{\nu}/N_{\nu}(t=0) - 1$')
plt.yscale("log")
plt.tight_layout()
plt.savefig(f"plots/{directory_path.split('/')[-1]}_n_ee_conservation_time.pdf")
plt.show()



# Compute errors

In [ ]:
def print_error(name, value):
    """Print a pasteable assignment. '-' in the run name becomes 'min'."""
    run = directory_path.split("/")[-1].replace("-", "min").replace(".", "p")
    print(f"{name}_{run} = {value}")

In [ ]:
# Discrete error measurements
n_i = nee[-1, :, 0, 0] # inv ccm
n_i_star = analytic_solution_all_times[-1, :] # inv ccm
e_i = n_i - n_i_star # inv ccm
q_i = n_i - n_bg_ccm # inv ccm
q_i_star = n_i_star - n_bg_ccm # inv ccm
V_omega_ccm = Lx*Ly*Lz # ccm
V_i_ccm = dx*dy*dz # ccm
Q_star = np.sum(q_i_star*V_i_ccm) # adimensional
A_star = np.max(q_i_star) # adimensional

# Maximum absolute and relative errors
E_infty = np.max(np.abs(e_i)) # adimensional
epsilon_infty = E_infty / A_star # adimensional
print_error("E_infty", E_infty)
print_error("epsilon_infty", epsilon_infty)

# Integrated absolute error
E_1 = np.sum(np.abs(e_i)*V_i_ccm) # adimensional
epsilon_1 = E_1 / np.sum(np.abs(q_i_star)*V_i_ccm) # adimensional
print_error("E_1", E_1)
print_error("epsilon_1", epsilon_1)

# L2 and RMS errors
E_2 = np.sqrt(np.sum(e_i**2*V_i_ccm)) # adimensional
E_RMS = np.sqrt(np.sum(e_i**2*V_i_ccm)/V_omega_ccm) # adimensional
epsilon_2 = E_2 / np.sqrt(np.sum(q_i_star**2*V_i_ccm)) # adimensional
print_error("E_2", E_2)
print_error("E_RMS", E_RMS)
print_error("epsilon_2", epsilon_2)

In [ ]:
###############################
# Numerical diffusion included
################################

E_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 1.669157730784471e+26
epsilon_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.006361964843064699
E_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 2.8385346667458073e+26
epsilon_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.005937098484046905
E_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 1.6141042139728044e+26
E_RMS_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 5.8436814604387944e+25
epsilon_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.0054196738777517964

E_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 2.6036815319498393e+26
epsilon_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.00992398694295719
E_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 4.496813083362876e+26
epsilon_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.00940556493906921
E_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 2.535162461627555e+26
E_RMS_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 9.178268508295294e+25
epsilon_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.008512308963819666

E_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 5.2183591710820785e+26
epsilon_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.019890466116226368
E_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 9.172344342863033e+26
epsilon_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.019184938034404925
E_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 5.123654666976469e+26
E_RMS_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 1.8549611312523074e+26
epsilon_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.017203683081209842

E_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 1.8069522020994113e+27
epsilon_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.06888522532971815
E_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 3.2889379013615804e+27
epsilon_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.06879164961920403
E_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 1.8048556417380967e+27
E_RMS_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 6.534275396279327e+26
epsilon_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.06060159492778404

E_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 5.497433588242202e+27
epsilon_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.20970708965363866
E_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 1.0841533889329293e+28
epsilon_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.22676226257010648
E_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 5.699845055147517e+27
E_RMS_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 2.0635643341861992e+27
epsilon_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.1913835617626123

###############################
# No numerical diffusion included
################################

# E_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 4.569734162834142e+25
# epsilon_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.0017498306049064886
# E_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 7.221150455240304e+25
# epsilon_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.001510380790867441
# E_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 4.30452598452464e+25
# E_RMS_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 1.5584048708869493e+25
# epsilon_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.0014486802371742862

# E_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 4.60281510881629e+25
# epsilon_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.001768823267307593
# E_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 7.297303056911871e+25
# epsilon_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.0015263089497042754
# E_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 4.343078692396875e+25
# E_RMS_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 1.5723624420457646e+25
# epsilon_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.001464268295748201

# E_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 4.685619567188168e+25
# epsilon_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.0018189164021180347
# E_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 7.499063553231595e+25
# epsilon_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.0015685094127648634
# E_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 4.442789549665867e+25
# E_RMS_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 1.6084616283921728e+25
# epsilon_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.0015054456965149904

# E_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 4.902864818521287e+25
# epsilon_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.0020033425095306624
# E_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 8.249039339905267e+25
# epsilon_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.0017253761213308749
# E_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 4.768986324500695e+25
# E_RMS_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 1.7265574755534593e+25
# epsilon_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.0016578166830843602

# E_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 4.278512877997534e+25
# epsilon_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.002060923672549788
# E_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 8.565125862318734e+25
# epsilon_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.0017915455846418268
# E_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 4.52847918123777e+25
# E_RMS_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 1.6394845888078498e+25
# epsilon_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.0017089133269730565


In [ ]:
dx_over_s0_cm = (Lx/s0_cm) / np.array([1024, 768, 512, 256, 128])
E_infty_all = [E_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, E_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, E_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, E_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, E_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
epsilon_infty_all = [epsilon_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, epsilon_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, epsilon_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, epsilon_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, epsilon_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
E_1_all = [E_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, E_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, E_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, E_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, E_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
epsilon_1_all = [epsilon_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, epsilon_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, epsilon_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, epsilon_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, epsilon_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
E_2_all = [E_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, E_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, E_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, E_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, E_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
E_RMS_all = [E_RMS_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, E_RMS_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, E_RMS_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, E_RMS_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, E_RMS_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
epsilon_2_all = [epsilon_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, epsilon_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, epsilon_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, epsilon_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, epsilon_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]

def linear_fit(x, y):
    """Least-squares fit of y = a*x + b. Returns (a, b)."""
    a, b = np.polyfit(np.asarray(x, dtype=float), np.asarray(y, dtype=float), 1)
    return float(a), float(b)

p_E_infty = linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_infty_all))[0]
p_epsilon_infty = linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_infty_all))[0]
p_E_1 = linear_fit(np.log10(dx_over_s0_cm), np.log10(E_1_all))[0]
p_epsilon_1 = linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_1_all))[0]
p_E_2 = linear_fit(np.log10(dx_over_s0_cm), np.log10(E_2_all))[0]
p_E_RMS = linear_fit(np.log10(dx_over_s0_cm), np.log10(E_RMS_all))[0]
p_epsilon_2 = linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_2_all))[0]

print(f"p_E_infty: {p_E_infty}")
print(f"p_epsilon_infty: {p_epsilon_infty}")
print(f"p_E_1: {p_E_1}")
print(f"p_epsilon_1: {p_epsilon_1}")
print(f"p_E_2: {p_E_2}")
print(f"p_E_RMS: {p_E_RMS}")
print(f"p_epsilon_2: {p_epsilon_2}")

C_E_infty = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_infty_all))[1])
C_epsilon_infty = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_infty_all))[1])
C_E_1 = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(E_1_all))[1])
C_epsilon_1 = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_1_all))[1])
C_E_2 = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(E_2_all))[1])
C_E_RMS = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(E_RMS_all))[1])
C_epsilon_2 = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_2_all))[1])

print(f"C_E_infty: {C_E_infty}")
print(f"C_epsilon_infty: {C_epsilon_infty}")
print(f"C_E_1: {C_E_1}")
print(f"C_epsilon_1: {C_epsilon_1}")
print(f"C_E_2: {C_E_2}")
print(f"C_E_RMS: {C_E_RMS}")
print(f"C_epsilon_2: {C_epsilon_2}")

In [ ]:
fontsize = 28
marker_size = 12
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.log10(dx_over_s0_cm), np.log10(epsilon_infty_all), label=r'$\epsilon_{\infty}\propto {\Delta x_{\mathrm{cell}}}^{' + f'{p_epsilon_infty:.3f}' + '}$', linewidth=4, marker='o', markersize=marker_size)
ax.plot(np.log10(dx_over_s0_cm), np.log10(epsilon_1_all), label=r'$\epsilon_1\propto {\Delta x_{\mathrm{cell}}}^{' + f'{p_epsilon_1:.3f}' + '}$', linewidth=4, marker='o', markersize=marker_size)
ax.plot(np.log10(dx_over_s0_cm), np.log10(epsilon_2_all), label=r'$\epsilon_2\propto {\Delta x_{\mathrm{cell}}}^{' + f'{p_epsilon_2:.3f}' + '}$', linewidth=4, marker='o', markersize=marker_size)
ax.set_ylabel(r'$\log_{10}\,\epsilon_i$', fontsize=fontsize)
# ax.set_xlim(np.log10(dx_over_s0_cm[0]), np.log10(dx_over_s0_cm[-1]))
ax.set_xlabel(r'$\log_{10}\,\,\Delta x_{\mathrm{cell}} / \sigma_0\,$', fontsize=fontsize)
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=fontsize)
pf.apply_custom_settings(ax, leg, False)
fig.tight_layout()
plt.show()